In [1]:
'''
    Calculated climos from history output
'''

'\n    Calculated climos from history output\n'

In [2]:
import xarray as xr
import numpy as np
#import xcdat as xcd

import glob as glob
import os as os
import re as re
import cftime as cft

In [3]:
from distributed import Client
from ncar_jobqueue import NCARCluster

cluster = NCARCluster(account='P93300042',interface='ext', job_extra_directives=[],walltime ='12:00:00')
cluster
cluster.scale(jobs=32)
client = Client(cluster)
client

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/dask_jobqueue/core.py:285: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/41075/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/41075/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.88:41027,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/41075/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [4]:
''' Case Details '''

#run_name = 'f.cam6_3_161.FLTHIST_ne30.ke.001'
run_name = 'f.cam6_4_032.FLTHIST_ne30.cam7_ke.005'

dir0_in = '/glade/derecho/scratch/rneale//archive/'
#dir0_in = '/glade/derecho/scratch/hannay/archive/'
#dir0_in = '/glade/derecho/scratch/gmarques/archive/'


dir0_out = '/glade/derecho/scratch/rneale/archive/'

hist_pref = 'h0a'
years = [1995,2005]



########################################




dir_in = dir0_in+run_name+'/atm/hist/'
dir_out = dir0_out+run_name+'/climo/'

syears = str(years)

# Create dir if needed.

if not os.path.exists(dir_out):
    os.makedirs(dir_out)
    print(f"Directory created: {dir_out}")
else:
    print(f"Directory already exists: {dir_out}")



# Sort files 

files_unsorted = os.path.join(dir_in, run_name+'*'+hist_pref+'*.nc')


# List all files matching the pattern
print(files_unsorted)
files_sorted = sorted(glob.glob(files_unsorted))
print('-List of all h0* files')
print(files_sorted[0])
print(files_sorted[-1])

print()
print('-List of all files for requested years - '+syears[0]+ ' to '+ syears[1])

# Subset files that are withing the year range of interest.

pattern = r'\b\d{4}\b'  # Matches exactly 4-digit numbers as whole words
files_in = []

for ff in files_sorted:
    # Find all 4-digit numbers in the string
    match = re.search(pattern, ff)
    if match:
        year = int(match.group())  # Convert to an integer (removes leading zeros)
        if years[0] <= year <= years[1]:
            files_in.append(ff)  # Add the string to the result if year is valid


#


#print(int(re.search(r'\d{4}', 'file_xx_1999').group()))

#files_in = [f for f in files_sorted if years[0] <= int(re.search(r'\d{4}', f).group()) <= years[1]]


print(files_in[0])
print(files_in[-1])

# File number check
print()
if len(files_in) != 12*(years[1]-years[0]+1): 
    print ('INCORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 
    sys.exit
else:
    print ('CORRECT NUMBER OF FILES FOR YEARS REQUESTED -- ',years[0],' to ',years[1]) 


Directory created: /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/
/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005*h0a*.nc
-List of all h0* files
/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005.cam.h0a.1995-01.nc
/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005.cam.h0a.2006-12.nc

-List of all files for requested years - [ to 1
/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005.cam.h0a.1995-01.nc
/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005.cam.h0a.2005-12.nc

CORRECT NUMBER OF FILES FOR YEARS REQUESTED --  1995  to  2005


In [5]:
'''
    Read in data (lazy)
'''

ds_hist = xr.open_mfdataset(files_in,parallel=True,chunks={"time": 12}) 

In [6]:

ds_hist.time

<xarray.DataArray 'time' (time: 132)> Size: 1kB
array([cftime.DatetimeNoLeap(1995, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 5, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 6, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 7, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 8, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 9, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 10, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1995, 12, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 5, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 6, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 7, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 8, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 9, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 10, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1996, 12, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 5, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 6, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 7, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 8, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 9, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 10, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1997, 12, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 5, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 6, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 7, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 8, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 9, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 10, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1998, 12, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1999, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1999, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1999, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1999, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cft

In [7]:
start_time = cft.DatetimeNoLeap(years[0], 1, 1)
end_time = cft.DatetimeNoLeap(years[1], 12, 31)

ds_hist = ds_hist.sel(time=slice(start_time, end_time))

In [8]:
'''
    Drop variables not needed and the string ones which cannot be grouped and averaged
'''

drop_vars_xtime = ['date_written','time_written','trop_cld_lev'] # Drop vars that are not time dimensioned, and add back in later if needed.

drop_vars_aer =['bc_c1','bc_c4','dst_c1','dst_c2','dst_c3','ncl_c1','ncl_c2','ncl_c3','num_c1','num_c2','num_c3','num_c4','pom_c1','pom_c4','so4_c1','so4_c2','so4_c3','soa_c1','soa_c2','bc_a1','bc_a4','dst_a1','dst_a2','dst_a3','ncl_a1','ncl_a2','ncl_a3','num_a1','num_a2','num_a3','so4_a1','so4_a2','so4_a3','soa_a1','soa_a2']

# Drop other unwanted 3D vars.
drop_vars_3d = ['ADRAIN','ADSNOW','ANSNOW','AWNI','AREI','AREL','AWNC','CCN3','CFC11','CFC12','CH4','CO2','DMS','GRAUQM','H2O2','H2SO4','N2O','NUMGRA','SNOWQM','SO2','SOAE','SOAG']

drop_vars_aer = [var for var in drop_vars_aer if var in ds_hist.data_vars]
ds_vars = ds_hist.drop_vars(drop_vars_aer)

drop_vars_3d = [var for var in drop_vars_3d if var in ds_hist.data_vars]
ds_vars = ds_vars.drop_vars(drop_vars_3d)

drop_vars_xtime = [var for var in drop_vars_xtime if var in ds_hist.data_vars]
ds_vars = ds_vars.drop_vars(drop_vars_xtime)

ds_vars.attrs["History Directory"] = dir_in
ds_vars.attrs["Start Year"] = str(years[0])
ds_vars.attrs["End Year"] = str(years[1])
    


ds_vars

<xarray.Dataset> Size: 159GB
Dimensions:           (time: 132, lat: 192, lev: 58, ilev: 59, nbnd: 2,
                       lon: 288, trop_pref: 58, trop_prefi: 59, trop_cld_lev: 58)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 464B 3.018 5.445 9.087 ... 983.2 991.2 997.5
  * ilev              (ilev) float64 472B 2.055 3.98 6.909 ... 987.4 995.1 1e+03
  * trop_pref         (trop_pref) float64 464B 3.018 5.445 9.087 ... 991.2 997.5
  * trop_prefi        (trop_prefi) float64 472B 2.055 3.98 6.909 ... 995.1 1e+03
  * trop_cld_lev      (trop_cld_lev) float64 464B 3.018 5.445 ... 991.2 997.5
  * time              (time) object 1kB 1995-01-16 12:00:00 ... 2005-12-16 12...
Dimensions without coordinates: nbnd
Data variables: (12/327)
    w                 (time, lat) float64 203kB dask.array<chunksize=(1, 192), meta=np.ndarray>
    hyam              (time, lev) float64 61kB dask.array<chunksize=(1, 58), meta=np.ndarray>
    hybm              (time, lev) float64 61kB dask.array<chunksize=(1, 58), meta=np.ndarray>
    hyai              (time, ilev) float64 62kB dask.array<chunksize=(1, 59), meta=np.ndarray>
    hybi              (time, ilev) float64 62kB dask.array<chunksize=(1, 59), meta=np.ndarray>
    date              (time) int32 528B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                ...
    RTP2_CLUBB        (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    THLP2_CLUBB       (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    RTPTHLP_CLUBB     (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    WPRCP_CLUBB       (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    WPTHVP_CLUBB      (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
    ZM_CLUBB          (time, ilev, lat, lon) float32 2GB dask.array<chunksize=(1, 59, 192, 288), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.cam6_4_032.FLTHIST_ne30.cam7_ke.005
    logname:            rneale
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/derecho/scratch/rneale//archive/f.cam6_4_032.F...
    Start Year:         1995
    End Year:           2005

In [9]:
'''
 Calculate monthly climo - easy
'''

ds_cmonth =  ds_vars.groupby("time.month").mean()

# Month names for output file.
mname_file = ["%02d" % x for x in ds_cmonth.month]
#ds_cmonth.mean(dim='month')
ds_cmonth
ds_cmonth.attrs

{'interp_type': 'bilinear',
 'interp_outputgri': 'equally spaced with poles',
 'Conventions': 'CF-1.0',
 'source': 'CAM',
 'case': 'f.cam6_4_032.FLTHIST_ne30.cam7_ke.005',
 'logname': 'rneale',
 'host': 'dec2356',
 'initial_file': '/glade/campaign/cesm/cesmdata/inputdata/atm/cam/inic/se/FLT_L58_ne30pg3_IC_c220623.nc',
 'topography_file': '/glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/se/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_20240117.nc',
 'model_doi_url': 'not_set',
 'time_period_freq': 'month_1',
 'History Directory': '/glade/derecho/scratch/rneale//archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/atm/hist/',
 'Start Year': '1995',
 'End Year': '2005'}

In [10]:
print('-Calculate annual climatology...')

ds_cyear =  ds_vars.mean(dim='time')

# Month names for output file.
#mname_file = ["%02d" % x for x in ds_cmonth.month]
ds_cyear

-Calculate annual climatology...


<xarray.Dataset> Size: 1GB
Dimensions:           (lat: 192, lev: 58, ilev: 59, lon: 288, trop_pref: 58,
                       trop_prefi: 59, trop_cld_lev: 58)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 464B 3.018 5.445 9.087 ... 983.2 991.2 997.5
  * ilev              (ilev) float64 472B 2.055 3.98 6.909 ... 987.4 995.1 1e+03
  * trop_pref         (trop_pref) float64 464B 3.018 5.445 9.087 ... 991.2 997.5
  * trop_prefi        (trop_prefi) float64 472B 2.055 3.98 6.909 ... 995.1 1e+03
  * trop_cld_lev      (trop_cld_lev) float64 464B 3.018 5.445 ... 991.2 997.5
Data variables: (12/326)
    w                 (lat) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    hyam              (lev) float64 464B dask.array<chunksize=(58,), meta=np.ndarray>
    hybm              (lev) float64 464B dask.array<chunksize=(58,), meta=np.ndarray>
    hyai              (ilev) float64 472B dask.array<chunksize=(59,), meta=np.ndarray>
    hybi              (ilev) float64 472B dask.array<chunksize=(59,), meta=np.ndarray>
    date              float64 8B dask.array<chunksize=(), meta=np.ndarray>
    ...                ...
    RTP2_CLUBB        (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    THLP2_CLUBB       (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    RTPTHLP_CLUBB     (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    WPRCP_CLUBB       (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    WPTHVP_CLUBB      (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>
    ZM_CLUBB          (ilev, lat, lon) float32 13MB dask.array<chunksize=(59, 192, 288), meta=np.ndarray>

In [11]:
print('-Calculate weighted seasonal climatology...')

days_in_month = ds_vars.time.dt.days_in_month

mweights = (days_in_month.groupby('time.season') / days_in_month.groupby('time.season').sum())

-Calculate weighted seasonal climatology...


In [12]:
# Calculate weighted seasonal averages

ds_wcseas = ((ds_vars*mweights).groupby("time.season").sum(dim='time'))
ds_wcseas = ds_wcseas.astype(np.float32)

In [13]:
# Need to copy the attributes due to the array multiplcation wiping them out.

# Global 

ds_wcseas.attrs = ds_vars.attrs 

# Variable

for var in ds_wcseas.data_vars:
    ds_wcseas[var].attrs = ds_vars[var].attrs



In [14]:
'''
    Writing out climo. files
'''

# Check out dir exists
if not os.path.exists(dir_out): os.makedirs(dir_out)

print('-Writing out to directory ...')

# Writing out monthly climatologies

for imm,mname in enumerate(ds_cmonth.month.values):
    
    fout_mon = dir_out+run_name+'_'+mname_file[imm]+'_climo.nc'
    print(mname_file[imm],' -- Writing - ',fout_mon)

    ds_cmonth.sel(month=mname).to_netcdf(fout_mon)
    print('-Done...')

# Writing out seasonal climatologies

-Writing out to directory ...
01  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_01_climo.nc
-Done...
02  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_02_climo.nc
-Done...
03  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_03_climo.nc
-Done...
04  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_04_climo.nc
-Done...
05  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_05_climo.nc
-Done...
06  -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_06_climo.nc
-Done...
07  -- Writing -

In [15]:
for sname in ds_wcseas.season.values:
    fout_seas = dir_out+run_name+'_'+sname+'_climo.nc'
    print(sname,'-- Writing - ',fout_seas)
    ds_wcseas.sel(season=sname).drop_vars('season').to_netcdf(fout_seas)
    print('-Done...')

DJF -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_DJF_climo.nc
-Done...
JJA -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_JJA_climo.nc
-Done...
MAM -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_MAM_climo.nc
-Done...
SON -- Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_SON_climo.nc
-Done...


In [16]:
# Wrting out annual climatology
fout_ann = dir_out+run_name+'_ANN_climo.nc'

print('-Writing - ',fout_ann)
ds_cyear.to_netcdf(fout_ann)
print('-Done...')

print('---- COMPLETE ----')

-Writing -  /glade/derecho/scratch/rneale/archive/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005/climo/f.cam6_4_032.FLTHIST_ne30.cam7_ke.005_ANN_climo.nc


/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 20.70 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


-Done...
---- COMPLETE ----
